# Seminar Notebook 3.4: Classification

**LSE MY459: Computational Text Analysis and Large Language Models** (WT 2026)

**Ryan Hübert**

This notebook covers supervised classification of texts using Naive Bayes, regularised regression, and cross-validation.

## Set up steps

### Directory management

We begin with some directory management to specify the file path to the folder on your computer where you wish to store data for this notebook.

In [ ]:
import os
sdir = os.path.join(os.path.expanduser("~"), "LSE-MY459-WT26", "SeminarWeek07")
dfm_path = os.path.join(sdir, "candidate-dfm.npz")
vocab_path = os.path.join(sdir, "candidate-dfm-features.txt")
corpus_path = os.path.join(sdir, "candidate-corpus.csv")
if any(not os.path.exists(x) for x in [dfm_path, vocab_path, corpus_path]):
    raise FileNotFoundError("You must run the first notebook before proceeding!")

### Loading the data

We will use the corpus of tweets posted by the major candidates for the 2016 U.S. presidential election, for which we created a DFM in `01-making-dfm.ipynb`. 

In [ ]:
import pandas as pd
import numpy as np
from scipy import sparse

# Load the sparse DFM
dfm = sparse.load_npz(dfm_path)
print(f"DFM shape: {dfm.shape}")

# Load the vocabulary
with open(vocab_path, "r") as f:
    vocabulary = f.read().split("\n")
print(f"Number of features: {len(vocabulary)}")

# Load the corpus metadata
corpus = pd.read_csv(corpus_path)
corpus.head()

## Labelling and splitting data

Our goal is to build a classifier that can predict whether tweets were posted by Hillary Clinton or not. This is a binary classification task, so we need to create a (binary) variable indicating whether each tweet was posted by Hillary Clinton. 

As discussed in lecture, we need to split our data into training and test sets. The training set is used to build the model, and the test set is held out to evaluate performance on unseen data. This helps us detect overfitting. We will use 75% for training and 25% for test.

## Classifying with Naive Bayes

Our first classifier is Naive Bayes. This model uses Bayes' theorem to calculate the probability of a document belonging to each class, assuming that words are conditionally independent given the class. For text with word counts, we use the Multinomial Naive Bayes model.

Using the estimated model, which we called `nb`, let's do prediction! First, we can extract the predicted probabilities.

Now we predict labels for both the training set (in-sample) and the test set (out-of-sample).

### Confusion matrices

We create confusion matrices to see how well our classifier performed. The rows represent true labels and columns represent predicted labels.

### Performance metrics

Let's create a function that calculates precision, recall, F1 and accuracy for a specific class. This helps us understand how well the classifier performs for each category.

### Using sklearn's built-in metrics

The `sklearn.metrics` module provides functions to calculate these metrics automatically. This is often more convenient than computing them by hand. That said, you should know how to calculate by hand.

Conveniently, `sklearn` also has a `classification_report()` function that provides all stats at once.

Notice that in-sample performance is typically better than out-of-sample performance. This is because the model has "seen" the training data and may have overfit to it.

### Tuning the smoothing parameter

Above we used `alpha=1.0` (standard Laplace smoothing), but this is a hyperparameter that can be tuned. Unlike ridge and lasso, sklearn doesn't provide a `MultinomialNBCV` function, so we use `GridSearchCV` to search over candidate values.

Compare with the original results above (alpha=1.0). Tuning often provides modest improvements, though for Naive Bayes the default Laplace smoothing is frequently near-optimal.

## Classifying with regularised regression

Another common approach to classification is regularised linear modelling, such as linear regression with a ridge or lasso penalty (this is the "regularisation"). Unlike Naive Bayes, these models directly estimate the relationship between features and the outcome. Regularisation penalises large coefficients, helping to prevent overfitting. For text data, it is often beneficial to apply TF-IDF weighting and/or normalisation. TF-IDF reduces the influence of very common words, while normalisation ensures that longer documents do not dominate the model. These transformations typically improve performance, though the optimal choice depends on the data.

Why didn't we use TF-IDF for Naive Bayes? Multinomial Naive Bayes has its own generative model for word counts, so it expects raw counts as input. Linear regression has no such built-in model for text, so rescaling the features with TF-IDF helps it work better.

Let's start with a ridge regression on our DFM. Recall that the optimal penalty ($\lambda$) should be chosen by cross-validation. In `sklearn`, `RidgeCV()` allows us to estimate a linear regression model with the ridge (L2) penalty, where the penalty is chosen via cross-validation. You first need to define the folds for cross-validation, and the values of $\lambda$ you want the algorithm to try. We'll do five fold cross-validation and search across 20 different values of $\lambda$.

What was the best penalty that the algorithm found?

Using the estimated model, which we called `ridge`, let's start doing prediction. First, we can extract the continuous predictions from the model, which we call "scores."

Usually we want to predict the _label_ for each document. We'll predict the labels for each document based on whether the score is greater than 0.5. We did not cover this in lecture, but the choice of the threshold could be optimised. Here, we use 0.5, but you can choose the threshold that optimises the performance statistics. You can learn more about this in a machine learning course.

Let's now examine the performance stats for this ridge regression.

### Other penalties 

As discussed in lecture, the ridge (L2) penalty is not the only option. An alternative is the lasso (L1) penalty. Unlike ridge, lasso can shrink some coefficients exactly to zero, effectively performing feature selection. A more general approach is the elastic net, which combines L1 and L2 penalties. In `sklearn`, `ElasticNetCV()` can be used to estimate lasso or elastic net with cross-validation. In the example below, we use the lasso penalty, but this can be extended to elastic net by choosing an `l1_ratio` between 0 and 1. 

Note: when I first ran this, I used `np.logspace(-4, 1, 100)` for the lambda grid, but the best lambda was at the boundary (the smallest value). This means the cross-validation didn't find an interior optimum, so I widened the grid to `np.logspace(-6, 1, 100)` to give the algorithm more room to search.

Now we extract both the scores and the predicted labels, just as we did above.

With lasso, some coefficients are set exactly to zero. Let's see how many features remain.

## Robust evaluation of classifiers with cross-validation

The single train/test split we used above is simple but has a limitation: our results depend on which documents happen to end up in each set. With a different random split, we might get different performance estimates. Cross-validation addresses this by repeatedly splitting the data, training on each split, and averaging results. This gives us more robust estimates of out-of-sample performance. Note: we already used cross-validation to select optimal hyperparameters for our regularised regression models. This was a slightly different use of cross-validation---i.e., using for _tuning_ the model, not evaluating it on held-out data.

In this example, we'll use a cross-validation approach with Naive Bayes classification, but this could also be done for regularised regression.

The `cross_val_predict` function above performs 10-fold cross-validation and returns predictions for every document. Crucially, each document's prediction comes from a model that was trained *without* that document. This gives us truly out-of-sample predictions for the entire dataset.

Let’s compare the performance statistics from 10-fold cross-validation and our original 25% test set. Cross-validation often yields slightly better performance estimates because each model is trained on a larger portion of the data and the results are averaged over multiple splits, reducing variability. In contrast, a single held-out test set reflects performance on one particular split and can therefore be more variable. However, there is no guarantee that cross-validation will produce better performance statistics than a test set. Moreover, because cross-validation is typically used for model selection, its performance estimates can be slightly optimistic. The held-out test set remains the most reliable measure of out-of-sample performance.

## Comparing classifiers

Let's compare all three classifiers side by side using the out-of-sample (test set) performance metrics for the Clinton class.

Notice the trade-offs between classifiers. Naive Bayes tends to have higher recall (it identifies more Clinton tweets) but lower precision (more false positives). Ridge and lasso have higher precision but lower recall. Which classifier is "best" depends on whether you care more about avoiding false positives (precision) or catching all true positives (recall). The F1 score provides a single summary that balances both.